In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [2]:
# Set the random seed for reproducibility
random.seed(0)
np.random.seed(0)

num_rows = 100  # Number of rows in the DataFrame

# Random dates for dataframe
start_date = datetime(2022, 1, 1)
end_date = datetime(2024, 12, 31)
date_range = [start_date + timedelta(days=random.randint(0, (end_date - start_date).days)) for _ in range(num_rows)]

# Random product IDs in a defined format (e.g., 'P001', 'P002', ..., 'P010')
product_ids = [f'P{str(random.randint(1, 10)).zfill(3)}' for _ in range(num_rows)]

# Random number of units sold (e.g., between 1 and 100)
units_sold = [random.randint(1, 100) for _ in range(num_rows)]

# Create the DataFrame
df = pd.DataFrame({
    'Date': date_range,
    'Product_ID': product_ids,
    'Units_Sold': units_sold
})

df.head()

,Date,Product_ID,Units_Sold
0,2024-02-28,P006,84
1,2024-05-11,P003,11
2,2022-03-24,P006,1
3,2023-06-15,P007,77
4,2024-11-13,P001,25


In [42]:
def unique_years(dataframe):
  """
  Find no of unique years in the DataFrame.

  Args:
    dataframe: Pandas DataFrame.

  Returns:
    Integer value representing unique years.
  """
  dataframe['Year'] = dataframe['Date'].dt.year
  unique_years = dataframe['Year'].nunique()

  return unique_years

print(unique_years(df))
df.head()

3


,Date,Product_ID,Units_Sold,Year
0,2024-02-28,P006,84,2024
1,2024-05-11,P003,11,2024
2,2022-03-24,P006,1,2022
3,2023-06-15,P007,77,2023
4,2024-11-13,P001,25,2024


In [26]:
def most_sold_product(df):
  """
  Find the year-wise most sold product.

  Args:
    df: Pandas DataFrame.

  Returns:
    A new dataframe with year, Product id of the most sold product and Units sold of that product.
  """
  # Group by 'Year' and 'Product_ID' and aggregate the total units sold
  yearly_sales = (
        df.groupby(['Year', 'Product_ID'])['Units_Sold']
          .sum()
          .reset_index()
    )

  sales_2023 = yearly_sales[yearly_sales['Year'] == 2023]

  # Find the most sold product for each year
  df_most_sold_per_year =  sales_2023.sort_values('Units_Sold',ascending=False).head(1)

  # Display the result
  return df_most_sold_per_year

most_sold_product(df)


,Year,Product_ID,Units_Sold
11,2023,P002,380


In [27]:
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()

In [28]:
print(housing['DESCR'])

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

In [30]:
housing_df = pd.DataFrame(housing['data'],columns=housing['feature_names'])
housing_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


In [35]:
# MedHouseVal expressed in hundreds of thousands of dollars ($100,000)
housing_df['MedHouseVal'] = housing['target']

In [36]:
def house_price_mmm(df,min_age,max_age):
  """
  Calculates mean and median of the MedhouseVal given the minimum and the maximum age limit of the house.

  Args:
    df: Pandas DataFrame.
    min_age: Minimum age of the house
    max_age: Maximum age of the house

  Returns:
    A tuple containing mean, and median.
  """
  new_df = df[(df['HouseAge'] >= min_age) & (df['HouseAge'] <= max_age)]
  mean = new_df['MedHouseVal'].mean()

  median = new_df['MedHouseVal'].median()


  return mean, median

mean_price, median_price = house_price_mmm(housing_df,20,30)

print("Mean age:", mean_price)
print("Median age:", median_price)


Mean age: 2.058271828098552
Median age: 1.841


In [38]:
housing_df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [40]:
import pandas as pd

def price_buckets(df, buckets, labels):
  """
  Add a column house price bucket which has the information about the price bucket in which the house falls.
  """

  # convert to actual dollar value
  max_house_price = df['MedHouseVal'] * 100000

  # create bucket column
  df['House_price_Bucket'] = pd.cut(
      max_house_price,
      bins=buckets + [float('inf')],
      labels=labels
  )

  # print counts of each bucket
  print(df['House_price_Bucket'].value_counts())

  return None


price_buckets(housing_df, [0, 50000, 100000], ['<50k', '50k-100k', '>100k'])

House_price_Bucket
>100k       16982
50k-100k     3448
<50k          210
Name: count, dtype: int64


In [41]:
def avg_rooms_per_person(df):
  """
  Add a "rooms_per_occup" column in the dataframe. This represents the number of rooms per occupant.

  Args:
    dataframe: Pandas DataFrame.

  Returns:
    The dataframe with an extra column "rooms_per_occup" added to it.
  """
  df['rooms_per_occup'] = df['AveRooms'] / df['AveOccup']
  maxval = df['rooms_per_occup'].max()

  return maxval

print(avg_rooms_per_person(housing_df))

55.22222222222222
